## Admin Workspace Information

> ⚠️ **This notebook is intended for instance admins only.**
> All data-fetching calls use the Workspaces **admin** API endpoints,
> which require the authenticated user
> to hold an **instance-admin** role. A dedicated check in **Section 1b** will
> halt execution if the account does not have admin rights.

This notebook provides a comprehensive admin view of all workspaces in an Evo instance using
`WorkspaceAdminHelper`, which wraps the admin endpoints of `WorkspaceAPIClient` together with
the File and Geoscience Object APIs.

### What this notebook covers

| Section | Description |
|---|---|
| **1a – Authentication** | Log in with OAuth and select your instance |
| **1b – Admin rights check** | Verify the user is an instance admin before proceeding |
| **2 – Workspace list** | All workspaces in the instance (via admin endpoint) |
| **3 – Users per workspace** | Per-workspace user list broken down by role (Owner / Editor / Viewer) |
| **4 – Objects per workspace** | Geoscience objects in each workspace grouped by schema type |
| **5 – Files per workspace** | Files in each workspace with size information |
| **6 – Summary report** | Aggregated one-row-per-workspace table for a quick admin overview |


## 1a. Authentication & Setup


In [ ]:
import os
import sys

# Make the helper module importable when running from this directory
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from evo.aio import AioTransport
from evo.common import APIConnector
from evo.oauth import AuthorizationCodeAuthorizer, EvoScopes, OAuthConnector
from evo.workspaces import WorkspaceAPIClient

# Evo app credentials
client_id = "<your-client-id>"  # Replace with your client ID
redirect_url = "<your-redirect-url>"  # Replace with your redirect URL

client_name = "Evo Admin Workspace Info Notebook"

transport = AioTransport(user_agent=client_name)

connector = OAuthConnector(
    transport=transport,
    client_id=client_id,
)

authorizer = AuthorizationCodeAuthorizer(
    oauth_connector=connector,
    redirect_url=redirect_url,
    scopes=EvoScopes.all_evo | EvoScopes.offline_access,
)

await authorizer.login()

### Select Instance

Use the widget below to pick your instance from the Discovery API.
Once a selection is made, run the next cell to create the API clients.


In [ ]:
from workspace_admin_helper import InstanceSelectorWidget

instance_selector = InstanceSelectorWidget(transport=transport, authorizer=authorizer)
await instance_selector.display()

### Create the API clients

> **NOTE:** `WorkspaceAdminHelper` handles all
> pagination and per-workspace client creation automatically.

In [ ]:
from workspace_admin_helper import WorkspaceAdminHelper

# Capture the selections made in the widget above.
org_id = instance_selector.get_org_id()
hub_url = instance_selector.get_hub_url()
hub_code = instance_selector.get_hub_code()
org_name = instance_selector.get_org_name()

if org_id is None or hub_url is None or hub_code is None:
    raise ValueError("Please select an instance from the widget above, then re-run this cell.")

api_connector = APIConnector(hub_url, transport, authorizer)

workspace_client = WorkspaceAPIClient(
    connector=api_connector,
    org_id=org_id,
)

# WorkspaceAdminHelper uses admin API endpoints, requiring the user to be an instance admin.
# hub_code is passed so the License Access admin-rights check is scoped to the correct hub.
admin_helper = WorkspaceAdminHelper(
    workspace_client=workspace_client,
    connector=api_connector,
    org_id=org_id,
    hub_code=hub_code,
)

print(f"Clients created for instance: {org_name} ({org_id})")
print(f"Hub: {hub_code}  ({hub_url})")

## 1b. Admin Rights Check

Before running any admin operations, verify that the authenticated user holds an
**instance-admin** role. A `403 Forbidden` response means the account is **not** an instance admin,
and execution is halted with a clear error message.

> If this cell raises `PermissionError`, ask your Evo administrator to grant the
> required instance-admin role to your account.


In [ ]:
is_admin = await admin_helper.check_admin_rights()

if not is_admin:
    raise PermissionError(
        "The authenticated user does not have org-admin rights.\n"
        "Admin API endpoints require an org-admin role.\n"
        "Please ask your Evo administrator to grant the required role."
    )

print("✅ Admin rights confirmed. You may proceed.")

## 2. List All Workspaces (Admin API)

Retrieves **all** workspaces in the instance using the admin endpoint. This works even for workspaces where the admin user has no direct role assigned.

In [ ]:
import pandas as pd

from evo.workspaces import Workspace

try:
    # Uses the admin endpoint — returns ALL workspaces in the instance
    all_workspaces: list[Workspace] = await admin_helper.list_workspaces()

    workspace_rows = [
        {
            "Name": ws.display_name,
            "ID": str(ws.id),
            # user_role is set when the admin has a direct role in the workspace,
            # None when they can see it only via admin rights
            "My Role": ws.user_role.name if ws.user_role else "—",
            "Member": "Yes" if ws.user_role is not None else "No",
            "Description": ws.description or "",
            "Labels": ", ".join(ws.labels) if ws.labels else "",
            "Coordinate System": ws.default_coordinate_system or "",
            "Created At": ws.created_at.strftime("%Y-%m-%d %H:%M"),
            "Updated At": ws.updated_at.strftime("%Y-%m-%d %H:%M"),
        }
        for ws in all_workspaces
    ]

    df_workspaces = pd.DataFrame(workspace_rows)
    member_count = sum(1 for ws in all_workspaces if ws.user_role is not None)
    print(
        f"Found {len(all_workspaces)} workspace(s) in the instance "
        f"({member_count} with a direct role, {len(all_workspaces) - member_count} admin-only)."
    )
    display(df_workspaces)
except Exception as e:
    print(f"Error listing workspaces: {e}")

## 3. Users Per Role Per Workspace (Admin API)

For each workspace, list the users and their roles using the admin endpoint.
This works even for workspaces where the admin user has no direct role assigned.


In [ ]:
from workspace_admin_helper import UserBrowserWidget

user_rows = []

for ws in all_workspaces:
    try:
        users = await admin_helper.get_users_for_workspace(ws.id)
        for user in users:
            user_rows.append(
                {
                    "Workspace": ws.display_name,
                    "Workspace ID": str(ws.id),
                    "Full Name": user.full_name or "",
                    "Email": user.email or "",
                    "User ID": str(user.user_id),
                    "Role": user.role.name if user.role else "unknown",
                }
            )
    except Exception as e:
        print(f"⚠ Could not fetch users for '{ws.display_name}': {e}")

if user_rows:
    df_users = pd.DataFrame(user_rows)
    UserBrowserWidget(df_users).show()
else:
    print("No user data available.")

### Role count summary across all workspaces

In [ ]:
if user_rows:
    df_role_summary = df_users.groupby(["Workspace", "Role"]).size().unstack(fill_value=0).reset_index()
    # Ensure all role columns exist
    for role in ("owner", "editor", "viewer"):
        if role not in df_role_summary.columns:
            df_role_summary[role] = 0

    df_role_summary.columns.name = None
    df_role_summary = df_role_summary[["Workspace", "owner", "editor", "viewer"]].rename(
        columns={"owner": "Owner", "editor": "Editor", "viewer": "Viewer"}
    )
    display(df_role_summary)
else:
    print("No user data available.")

## 4. Geoscience Objects Per Workspace

Fetch all geoscience objects in each workspace and group them by schema type.
Only workspaces where the authenticated user has a **direct role** are queried,
since object access requires an active workspace membership.

> **Note:** If a workspace you need is not listed, add yourself as a member of that workspace
> in Evo before re-running this section.


In [ ]:
object_rows = []
object_errors = []

# Only query workspaces where the user has a direct role assigned.
member_workspaces = [ws for ws in all_workspaces if ws.user_role is not None]
skipped = len(all_workspaces) - len(member_workspaces)
if skipped:
    print(f"Skipping {skipped} workspace(s) where the user has no direct role.")

for ws in member_workspaces:
    try:
        objects = await admin_helper.get_objects_for_workspace(ws)
        for obj in objects:
            object_rows.append(
                {
                    "Workspace": ws.display_name,
                    "Object Name": obj.name,
                    "Object ID": str(obj.id),
                    "Schema": str(obj.schema_id),
                    "Path": obj.path,
                    "Modified At": obj.modified_at.strftime("%Y-%m-%d %H:%M") if obj.modified_at else "",
                    "Modified By": obj.modified_by.name if obj.modified_by else "",
                }
            )
    except Exception as e:
        object_errors.append({"Workspace": ws.display_name, "Error": str(e)})

if object_errors:
    print("Errors fetching objects:")
    display(pd.DataFrame(object_errors))

if object_rows:
    df_objects = pd.DataFrame(object_rows)
    print(f"\nTotal objects found: {len(df_objects)} across {len(member_workspaces)} workspace(s).")
    display(df_objects)
else:
    print("No objects found (or errors occurred for all member workspaces).")

### Object count by schema type per workspace

In [ ]:
if object_rows:
    df_object_schema = (
        df_objects.groupby(["Workspace", "Schema"])
        .size()
        .rename("Count")
        .reset_index()
        .sort_values(["Workspace", "Count"], ascending=[True, False])
    )
    display(df_object_schema.reset_index(drop=True))
else:
    print("No object data available.")

## 5. Files Per Workspace

Fetch all files in each workspace including path, size, and modification details.
Only workspaces where the authenticated user has a **direct role** are queried,
since file access requires an active workspace membership.

> **Note:** If a workspace you need is not listed, add yourself as a member of that workspace
> in Evo before re-running this section.


In [ ]:
from workspace_admin_helper import format_size

file_rows = []
file_errors = []

# Only query workspaces where the user has a direct role assigned.
file_workspaces = [ws for ws in all_workspaces if ws.user_role is not None]
skipped_files = len(all_workspaces) - len(file_workspaces)
if skipped_files:
    print(f"Skipping {skipped_files} workspace(s) where the user has no direct role.")

for ws in file_workspaces:
    try:
        files = await admin_helper.get_files_for_workspace(ws)
        for f in files:
            file_rows.append(
                {
                    "Workspace": ws.display_name,
                    "File Name": f.name,
                    "File ID": str(f.id),
                    "Path": f.path,
                    "Size (bytes)": f.size,
                    "Modified At": f.modified_at.strftime("%Y-%m-%d %H:%M") if f.modified_at else "",
                    "Modified By": f.modified_by.name if f.modified_by else "",
                }
            )
    except Exception as e:
        file_errors.append({"Workspace": ws.display_name, "Error": str(e)})

if file_errors:
    print("Errors fetching files:")
    display(pd.DataFrame(file_errors))

if file_rows:
    df_files = pd.DataFrame(file_rows)
    df_files.insert(
        df_files.columns.get_loc("Size (bytes)") + 1,
        "Size",
        df_files["Size (bytes)"].apply(format_size),
    )
    print(f"\nTotal files found: {len(df_files)} across {len(file_workspaces)} workspace(s).")
    display(df_files.drop(columns=["Size (bytes)"]))
else:
    print("No files found (or errors occurred for all member workspaces).")

### File storage summary per workspace

In [ ]:
if file_rows:
    df_file_summary = (
        df_files.groupby("Workspace")
        .agg(
            File_Count=("File Name", "count"),
            Total_Size_bytes=("Size (bytes)", "sum"),
        )
        .reset_index()
        .rename(columns={"File_Count": "File Count"})
        .sort_values("Total_Size_bytes", ascending=False)
    )
    df_file_summary["Total Size"] = df_file_summary["Total_Size_bytes"].apply(format_size)
    df_file_summary = df_file_summary.drop(columns=["Total_Size_bytes"])
    display(df_file_summary.reset_index(drop=True))
else:
    print("No file data available.")

## 6. Full Admin Summary Report

Builds a single aggregated table with one row per workspace.
Only workspaces where the authenticated user has a **direct role** are included,
since object and file access requires an active workspace membership.

> **Note:** If a workspace you need is not listed, add yourself as a member of that workspace
> in Evo before re-running this section.


In [ ]:
from workspace_admin_helper import WorkspaceReport

# Restrict the report to workspaces where the user has a direct role.
member_workspace_ids = [ws.id for ws in all_workspaces if ws.user_role is not None]

try:
    reports: list[WorkspaceReport] = await admin_helper.get_all_workspace_reports(
        include_objects=True,
        include_files=True,
        workspace_ids=member_workspace_ids,
    )

    summary_rows = [r.summary_dict() for r in reports]
    df_summary = pd.DataFrame(summary_rows)

    print(f"Admin summary for {len(reports)} workspace(s) (member workspaces only):")
    display(df_summary)
except Exception as e:
    print(f"Error generating summary report: {e}")

## 7. Export to Excel

Export all report data to a single `.xlsx` file with one sheet per data section.
The file is saved in the same directory as this notebook.

| Sheet | Contents |
|---|---|
| **Workspaces** | All workspaces with role and membership status |
| **Users** | Per-workspace user list with roles |
| **Role Summary** | User count by role per workspace |
| **Objects** | All geoscience objects (member workspaces only) |
| **Object Schema Summary** | Object count grouped by schema type per workspace |
| **Files** | All files with size info (member workspaces only) |
| **File Storage Summary** | Total file count and storage per workspace |
| **Admin Summary** | One-row-per-workspace aggregated report |


In [ ]:
import os
from datetime import datetime

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(notebook_dir, f"workspace_admin_report_{timestamp}.xlsx")

_sheet_map = [
    ("Workspaces", "df_workspaces"),
    ("Users", "df_users"),
    ("Role Summary", "df_role_summary"),
    ("Objects", "df_objects"),
    ("Object Schema Summary", "df_object_schema"),
    ("Files", "df_files"),
    ("File Storage Summary", "df_file_summary"),
    ("Admin Summary", "df_summary"),
]

written = []
with pd.ExcelWriter(output_path) as writer:
    for sheet_name, var_name in _sheet_map:
        df = globals().get(var_name)
        if df is None or df.empty:
            print(f"Skipping '{sheet_name}' — no data available.")
            continue
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        written.append(sheet_name)

if written:
    print(f"Exported {len(written)} sheet(s) to:\n  {output_path}")
else:
    print("Nothing to export.")

## Instance-level User Management

The `WorkspaceAPIClient` also exposes instance-level user operations. The cell below show
how to list all users and roles at the instance level.


In [ ]:
from evo.workspaces import InstanceUserWithEmail

try:
    # list_instance_users returns a Page; retrieve the first page (up to 50 users)
    page = await workspace_client.list_instance_users(limit=50, offset=0)
    instance_users: list[InstanceUserWithEmail] = page.items()

    instance_user_rows = [
        {
            "User ID": str(u.user_id),
            "Email": u.email,
            "Full Name": u.full_name,
            "Instance Roles": ", ".join(r.name for r in u.roles),
        }
        for u in instance_users
    ]

    print(f"Instance users on this page: {len(instance_users)} (total: {page.total})")
    display(pd.DataFrame(instance_user_rows))
except Exception as e:
    print(f"Error listing instance users: {e}")